In [1]:
import gc
import os
import sys

import numpy as np
import optuna
import pandas as pd
import torch
from sklearn.model_selection import KFold

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [14]:
%load_ext autoreload
%autoreload 2
from drGAT import drGAT
from drGAT.load_data import load_data
from drGAT.sampler import RandomSampler

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
drugAct, pos_num, null_mask, S_d, S_c, S_g, A_cg, A_dg = load_data("gdsc2")

load gdsc2
Index(['5-Fluorouracil', '5-azacytidine', 'A-366', 'ABT737', 'AGI-5198',
       'AGI-6780', 'AGK2', 'AMG-319', 'AT13148', 'AZ6102',
       ...
       'WZ4003', 'Wee1 Inhibitor', 'Wnt-C59', 'XAV939', 'YK-4-279', 'ZM447439',
       'Zoledronate', 'alpha-lipoic acid', 'ascorbate (vitamin C)',
       'glutathione'],
      dtype='object', length=240)
unique drugs: 69
unique genes: 153
DTI unique genes:  153
Top 90% variable genes:  1957
Total:  2089
Final gene exp shape: (910, 2089)
Final drug Act shape: (240, 910)


100%|██████████| 25/25 [00:03<00:00,  7.30it/s]


Done!


In [ ]:
PATH = "../gdsc2_data/"

In [19]:
def objective(trial):
    params = {
        "n_drug": S_d.shape[0],
        "n_cell": S_c.shape[0],
        "n_gene": S_g.shape[0],
        "dropout1": trial.suggest_categorical("dropout1", [0.1, 0.2, 0.3, 0.4, 0.5]),
        "dropout2": trial.suggest_categorical("dropout2", [0.1, 0.2, 0.3, 0.4, 0.5]),
        "hidden1": trial.suggest_categorical(
            "hidden1",
            [256, 512, 1028],
        ),
        "hidden2": trial.suggest_categorical(
            "hidden2",
            [
                128,
                256,
                512,
            ],
        ),
        "hidden3": trial.suggest_categorical(
            "hidden3",
            [
                64,
                128,
                256,
            ],
        ),
        "epochs": trial.suggest_categorical("epochs", [10, 50, 100, 200, 500]),
        "heads": trial.suggest_categorical("heads", [1, 2, 3, 4, 5]),
        "activation": trial.suggest_categorical(
            "activation", ["relu", "gelu", "swish"]
        ),
        "optimizer": trial.suggest_categorical("optimizer", ["Adam", "AdamW", "SGD"]),
        "lr": trial.suggest_float("lr", 1e-5, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        "scheduler": trial.suggest_categorical(
            "scheduler", [None, "Cosine", "Step", "Plateau"]
        ),
        "gnn_layer": trial.suggest_categorical(
            "gnn_layer", ["GAT", "GATv2", "Transformer"]
        ),
    }

    # スケジューラ関連パラメータの条件付き追加
    if params["scheduler"] == "Cosine":
        params["T_max"] = trial.suggest_int("T_max", 20, 50)
    elif params["scheduler"] == "Step":
        params["scheduler_gamma"] = trial.suggest_float("gamma_step", 0.1, 0.95)
        params["step_size"] = trial.suggest_int("step_size", 10, 30)
    elif params["scheduler"] == "Plateau":
        params["patience"] = trial.suggest_int("patience_plateau", 3, 10)
        params["threshold"] = trial.suggest_float(
            "thresh_plateau", 1e-4, 1e-2, log=True
        )

    if params["hidden1"] < params["hidden2"] or params["hidden2"] < params["hidden3"]:
        raise optuna.TrialPruned("Invalid layer size configuration")

    if params["optimizer"] in ["Adam", "AdamW"]:
        params["amsgrad"] = trial.suggest_categorical("amsgrad", [True, False])

    if params["optimizer"] == "SGD":
        params["momentum"] = trial.suggest_float("momentum", 0.8, 0.95)
        params["nesterov"] = trial.suggest_categorical("nesterov", [True, False])

    # 隠れ層サイズとバッチサイズの関係を制約
    if (params["hidden1"] > 512) and (params["hidden2"] > 256):
        raise optuna.TrialPruned("Memory intensive configuration")

    try:
        k = 5
        kfold = KFold(n_splits=k, shuffle=True, random_state=42)

        res = pd.DataFrame()
        for train_index, test_index in kfold.split(np.arange(pos_num)):
            sampler = RandomSampler(
                drugAct,
                train_index,
                test_index,
                null_mask,
                S_d,
                S_c,
                S_g,
                A_cg,
                A_dg,
                PATH,
            )
            (_, _, _, best_metrics, _, _, _) = drGAT.train(
                sampler, params=params, device=device, verbose=False
            )

            res = pd.concat(
                [
                    res,
                    pd.DataFrame(best_metrics, index=["acc", "f1", "auroc", "aupr"]).T,
                ]
            )

        return [float(i) for i in res.mean()]

    except RuntimeError as e:
        if "CUDA out of memory" in str(e):
            print(f"Pruned trial {trial.number}: CUDA OOM")

            # メモリ解放処理
            with torch.cuda.device("cuda"):
                torch.cuda.empty_cache()
            gc.collect()

            # Pruning通知
            raise optuna.TrialPruned(f"OOM at trial {trial.number}")

        else:
            raise e

In [20]:
name = "GDSC2_GAT"
study = optuna.create_study(
    directions=["maximize"] * 4,
    sampler=optuna.samplers.TPESampler(),
    pruner=optuna.pruners.HyperbandPruner(),
    storage="sqlite:///{}.sqlite3".format(name),
    study_name=name,
    load_if_exists=True,
)
study.optimize(objective, n_trials=100)

[I 2025-03-21 18:01:38,462] A new study created in RDB with name: GDSC1_GAT
/Users/inouey2/code/drGAT/.venv/lib/python3.12/site-packages/optuna/distributions.py:699: UserWarning: The distribution is specified by [10, 200] and step=50, but the range is not divisible by `step`. It will be replaced by [10, 160].
  warnings.warn(


Using:  cpu


/Users/inouey2/code/drGAT/.venv/lib/python3.12/site-packages/torch/amp/grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(
  0%|                                                                                                  | 0/10 [00:08<?, ?it/s]
[W 2025-03-21 18:01:47,292] Trial 0 failed with parameters: {'dropout1': 0.4, 'dropout2': 0.1, 'hidden1': 256, 'hidden2': 256, 'hidden3': 64, 'epochs': 10, 'heads': 3, 'activation': 'gelu', 'optimizer': 'AdamW', 'lr': 0.0003883278672260954, 'weight_decay': 0.0001509324807739926, 'scheduler': 'Plateau', 'gnn_layer': 'Transformer', 'patience_plateau': 5, 'thresh_plateau': 0.0001662001761327504, 'amsgrad': False} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Users/inouey2/code/drGAT/.venv/lib/python3.12/site-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                  

KeyboardInterrupt: 

## Eval

In [ ]:
# test_drug = test_data.values[:, 0]
# test_cell = test_data.values[:, 1]

# test_labels = np.load("data/test_labels.npy")
# test_labels = torch.tensor(test_labels).float()
# test = [drug, cell, gene, edge_index, test_drug, test_cell, test_labels]

In [22]:
# prob, res, test_attention = drGAT.eval(model, test)